# Extract High-mass XRB


### A catalogue of high-mass X-ray binaries in the Galaxy: from the INTEGRAL to the Gaia era 
### Fortin et al. 2023

2023A&A...671A.149F


https://ui.adsabs.harvard.edu/abs/2023A%26A...671A.149F/abstract

**They list the HMXB on their on website:)** 
https://binary-revolution.github.io/HMXBwebcat/

We use their latest version:
v2024-08



In [15]:
# Lets start with some imports
import pandas as pd
import numpy as np
import re
import json
import sys
from pathlib import Path
from textwrap import dedent

proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

# -------------------------------------------------
# Configuration
# -------------------------------------------------
from paths import RESULT_TABLES, RAW_JSON_DIR, DATA_DIR

## Load their convenient csv format table

More info on the columns:

subclass of the HMXBs: Be, supergiant (sg), supergiant fast X-ray transient (SFXT), and a few peculiar sub-classes such as sgB[e] or Wolf-Rayet (WR)


variability flag (“Var”) that summarises whether the
HMXBs were flagged as variable sources in the INTEGRAL, 4XMM DR11, or Chandra catalogues, or if the ratio of the peak
to mean flux in the Swift 2SXPS catalogue is greater than 5


mass of the compact object (Mx) and the companion star (Mo). Companion masses that were broadly inferred from the spectral type by us are labelled with a dagger

In [16]:
table_file = Path(DATA_DIR) / 'from_others' / 'HMXBwebcat_latest.csv'
df = pd.read_csv(table_file, sep=',')
display(df)
    

,Main_ID,RAdeg,DEdeg,PosErr,Pos_ref,Spectype,Spectype_ref,Class,Compact,Mx,...,2MASS_ID,2MASS_RA,2MASS_Dec,2MASS_err,2MASS_ref,Gaia_ID,Gaia_RA,Gaia_Dec,Gaia_err,Gaia_ref
0,IGR J00370+6122,9.290133,61.360133,2.337165e-09,NaN,BN0.7 Ib,2014A&A...566A.131G,sg,NS,NaN,...,2MASS J00370963+6121363,9.290125,61.360111,0.000019,NaN,427234969757165952,9.290133,61.360133,2.337165e-09,NaN
1,gam Cas,14.177451,60.716723,5.093332e-07,NaN,B0.5IVpe,2011ARep...55...31S,Be,NaN,NaN,...,2MASS J00564251+6043002,14.177127,60.716743,0.000081,NaN,426558460884582016,14.177451,60.716723,5.093332e-07,NaN
2,EM* AS 14,18.996041,59.153945,3.050714e-09,NaN,B2,1960IzKry..24..160B,NaN,NaN,NaN,...,2MASS J01155905+5909141,18.996062,59.153919,0.000017,NaN,414196617287885312,18.996041,59.153945,3.050714e-09,NaN
3,2S 0114+650,19.511227,65.291623,2.072556e-09,NaN,B1Iae,2015A&A...579A.111K,sg,NS,NaN,...,2MASS J01180266+6517298,19.511102,65.291618,0.000017,NaN,524924310153249920,19.511227,65.291623,2.072556e-09,NaN
4,4U 0115+634,19.633193,63.742522,3.026561e-09,NaN,B0.2Ve,2001A&A...369..108N,Be,NS,NaN,...,2MASS J01183196+6344330,19.633192,63.742519,0.000017,NaN,524677469790488960,19.633193,63.742522,3.026561e-09,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,1H 2202+501,330.409192,50.167952,3.113065e-09,NaN,Be,1964LS....C03....0H,Be,NaN,NaN,...,2MASS J22013820+5010046,330.409184,50.167953,0.000017,NaN,1979911002134040960,330.409192,50.167952,3.113065e-09,NaN
160,4U 2206+543,331.984288,54.518432,3.347356e-09,NaN,O9.5Vep,2006A&A...446.1095B,Be,NS,NaN,...,2MASS J22075623+5431064,331.984313,54.518459,0.000019,NaN,2005653524280214400,331.984288,54.518432,3.347356e-09,NaN
161,SAX J2239.3+6116,339.836830,61.274053,3.460414e-09,NaN,B0Ve,2017A&A...598A..16R,Be,NS,NaN,...,2MASS J22392085+6116266,339.836886,61.274059,0.000019,NaN,2201091578667140352,339.836830,61.274053,3.460414e-09,NaN
162,MWC 656,340.738741,44.721725,3.678116e-09,NaN,B1.5-B2IIIe,2014Natur.505..378C,Be,BH,5.4,...,2MASS J22425730+4443183,340.738777,44.721760,0.000017,NaN,1982359580155628160,340.738741,44.721725,3.678116e-09,NaN


In [17]:
display(df[['Period','Period_err','Eccentricity','Eccentricity_err']][df['Main_ID'] == 'EXO 1722-363'])

,Period,Period_err,Eccentricity,Eccentricity_err
87,9.7403,0.0004,0.19,-1.0


In [18]:
print([key for key in df.keys()]    )

['Main_ID', 'RAdeg', 'DEdeg', 'PosErr', 'Pos_ref', 'Spectype', 'Spectype_ref', 'Class', 'Compact', 'Mx', 'Mx_err', 'Mx_ref', 'Mo', 'Mo_err', 'Mo_ref', 'Period', 'Period_err', 'Period_ref', 'Superorbital Period', 'Superorbital Period_err', 'Superorbital Period_ref', 'Eccentricity', 'Eccentricity_err', 'Eccentricity_ref', 'Spin_period', 'Spin_period_err', 'Spin_period_ref', 'RV', 'RV_err', 'RV_ref', 'Distance', 'Distance_err_lo', 'Distance_err_up', 'IGR_var', 'Swift_var', 'XMM_var', 'Chandra_var', 'Var', 'best_ID', 'AGILE_ID', 'AGILE_RA', 'AGILE_Dec', 'AGILE_err', 'AGILE_ref', 'HEAO_ID', 'HEAO_RA', 'HEAO_Dec', 'HEAO_err', 'HEAO_ref', 'UHURU4_ID', 'UHURU4_RA', 'UHURU4_Dec', 'UHURU4_err', 'UHURU4_ref', 'ARIEL3_ID', 'ARIEL3_RA', 'ARIEL3_Dec', 'ARIEL3_err', 'ARIEL3_ref', 'IGR_ID', 'IGR_RA', 'IGR_Dec', 'IGR_err', 'IGR_ref', '2E_ID', '2E_RA', '2E_Dec', '2E_err', '2E_ref', 'ROSAT_ID', 'ROSAT_RA', 'ROSAT_Dec', 'ROSAT_err', 'ROSAT_ref', 'ROSATF_ID', 'ROSATF_RA', 'ROSATF_Dec', 'ROSATF_err', 'ROSAT

In [19]:
display(df[['Main_ID','Class','Period', 'Eccentricity', 'Compact','Mx', 'Mo', 'PosErr','Spectype']][df['Mx'].isna()])

display(df[['Main_ID','Class','Period', 'Eccentricity', 'Compact','Mx', 'Mo', 'PosErr','Spectype']][np.logical_and(df['Mx'].isna()== False, df['Mx']> 2.5)])
print(sum(np.logical_and(df['Mx'].isna()== False, df['Mx']> 2.5)))

display(df[['Main_ID','Class','Period', 'Eccentricity', 'Compact','Mx', 'Mo', 'PosErr','Spectype']][np.logical_and(df['Mx'].isna()== False, df['Mx']< 2.5)])
print(sum(np.logical_and(df['Mx'].isna()== False, df['Mx']< 2.5)))


,Main_ID,Class,Period,Eccentricity,Compact,Mx,Mo,PosErr,Spectype
0,IGR J00370+6122,sg,15.6649,0.480,NS,NaN,22.0,2.337165e-09,BN0.7 Ib
1,gam Cas,Be,203.3710,0.260,NaN,NaN,13.0,5.093332e-07,B0.5IVpe
2,EM* AS 14,NaN,NaN,NaN,NaN,NaN,NaN,3.050714e-09,B2
3,2S 0114+650,sg,11.5983,0.180,NS,NaN,16.0,2.072556e-09,B1Iae
4,4U 0115+634,Be,24.3174,0.339,NS,NaN,17.5,3.026561e-09,B0.2Ve
...,...,...,...,...,...,...,...,...,...
158,Cep X-4,Be,20.8500,NaN,NS,NaN,10.8,3.202740e-09,B1-B2Ve
159,1H 2202+501,Be,NaN,NaN,NaN,NaN,NaN,3.113065e-09,Be
160,4U 2206+543,Be,9.5580,0.300,NS,NaN,18.0,3.347356e-09,O9.5Vep
161,SAX J2239.3+6116,Be,262.0000,NaN,NS,NaN,17.5,3.460414e-09,B0Ve


,Main_ID,Class,Period,Eccentricity,Compact,Mx,Mo,PosErr,Spectype
36,HD 96670,Be,5.283880,0.12,BH,6.2,22.7,6.412144e-09,O7V(f)n
93,1E 1740.7-2942,NaN,12.610000,NaN,BH,5.0,NaN,6.059417e-05,NaN
107,SAX J1819.3-2525,NaN,2.817000,NaN,BH,10.2,6.8,5.672918e-09,B9III
138,SS 433,sg,13.080000,0.05,BH,4.2,11.3,5.110105e-09,A7Ib
147,Cyg X-1,sg,5.599800,0.00,NaN,21.2,40.6,3.038323e-09,O9.7Iabpvar
154,Cyg X-3,WR,0.199685,NaN,NaN,7.2,NaN,1.666667e-05,WN4/5-6/7
162,MWC 656,Be,60.370000,0.10,BH,5.4,7.8,3.678116e-09,B1.5-B2IIIe


7


,Main_ID,Class,Period,Eccentricity,Compact,Mx,Mo,PosErr,Spectype
18,HD 259440,"$\gamma$\,Be",317.30000,0.6200,NS?,1.40,15.7,4.864935e-09,B0pe
24,SGR 0755-2933,Be,59.69000,0.0600,NS,1.40,18.5,2.420969e-09,B0Ve
30,Vela X-1,sg,8.96302,0.1070,NS,2.12,26.0,3.016724e-09,B0.5Iae-1b
34,1FGL J1018.6-5856,"$\gamma$\,Be",16.55070,0.5310,NaN,2.00,22.9,2.595992e-09,O6V
38,Cen X-3,sg,2.03300,0.0001,NS,1.34,20.2,3.231639e-09,O6-7 II-III
42,1E 1145.1-6141,sg,14.36500,0.2000,NS,1.70,14.0,2.094991e-09,B2Iae
63,4U 1538-522,sg,3.72831,0.1800,NS,1.18,20.0,3.029035e-09,B0.2Ia
83,OAO 1657-415,WR,10.44812,0.1070,NS,1.42,14.3,1.666667e-05,Ofpe/WN9
84,4U 1700-377,sg,3.41166,0.0300,NaN,1.96,46.0,5.847068e-09,O6Iafcp
87,EXO 1722-363,sg,9.74030,0.1900,NS,1.91,18.0,1.666667e-05,B0-1Ia


12


In [20]:
nan_Compac = df['Compact'].isna()
nan_Mx = df['Mx'].isna()
# display(df[['Mo', 'Mx', 'Compact']])
display(df[['Mo', 'Mx', 'Compact']][~nan_Compac])
display(df[['Period','Eccentricity','Mo', 'Mx', 'Compact']][~nan_Mx])



,Mo,Mx,Compact
0,22.0,NaN,NS
3,16.0,NaN,NS
4,17.5,NaN,NS
6,9.6,NaN,NS
7,12.5,NaN,NS
...,...,...,...
158,10.8,NaN,NS
160,18.0,NaN,NS
161,17.5,NaN,NS
162,7.8,5.4,BH


,Period,Eccentricity,Mo,Mx,Compact
18,317.300000,0.6200,15.7,1.40,NS?
24,59.690000,0.0600,18.5,1.40,NS
30,8.963020,0.1070,26.0,2.12,NS
34,16.550700,0.5310,22.9,2.00,NaN
36,5.283880,0.1200,22.7,6.20,BH
38,2.033000,0.0001,20.2,1.34,NS
42,14.365000,0.2000,14.0,1.70,NS
63,3.728310,0.1800,20.0,1.18,NS
83,10.448120,0.1070,14.3,1.42,NS
84,3.411660,0.0300,46.0,1.96,NaN


# Now pour this into our json data_schema

In [21]:
# Helper function to create triplets with uncertainties
def make_triplet(value, err_lo=None, err_up=None):
    """Create [err-, value, err+] triplet with absolute errors, not bounds."""
    if pd.isna(value):
        return [None, None, None]
    
    val = float(value)
    
    # Handle symmetric errors
    if pd.notna(err_lo) and pd.isna(err_up):
        err = float(err_lo) # absolute error (there are a few off -1 values in the catalog)
        if err == -1:
            err = 0 # this indicates no error
        return [err, val, err]
    # Handle asymmetric errors
    elif pd.notna(err_lo) and pd.notna(err_up):
        return [abs(float(err_lo)), val, abs(float(err_up))]
    # No error bars
    else:
        return [None, val, None]

def collect_references(row):
    """Collect all non-null reference values from *_ref columns."""
    ref_cols = [col for col in row.index if col.endswith('_ref')]
    refs = []
    for col in ref_cols:
        val = row[col]
        if pd.notna(val) and val not in (None, '', 'nan'):
            # Split by comma or semicolon if multiple references in one field
            if isinstance(val, str):
                for ref in val.replace(';', ',').split(','):
                    ref = ref.strip()
                    if ref and ref not in refs:
                        refs.append(ref)
            else:
                refs.append(str(val))
    return refs if refs else ["2023A&A...671A.149F"]  # Default to catalog paper

def map_compact_to_evol_type(compact_str):
    """Map compact object type to evolutionary type."""
    if pd.isna(compact_str):
        return None
    c = str(compact_str).strip().upper()
    if 'NS' in c or 'NEUTRON' in c or 'PULSAR' in c:
        return 'NS'
    elif 'BH' in c or 'BLACK HOLE' in c:
        return 'BH'
    elif 'WD' in c or 'WHITE DWARF' in c:
        return 'WD'
    else:
        return None  # Unknown

print("Mapping HMXB catalog to JSON schema format...")
print(f"Total systems in catalog: {len(df)}")

Mapping HMXB catalog to JSON schema format...
Total systems in catalog: 164


In [22]:
# Convert DataFrame to schema format
systems = []
for idx, row in df.iterrows():
    # System name from Main_ID
    system_name = str(row['Main_ID']) if pd.notna(row.get('Main_ID')) else f"HMXB_{idx}"
    
    # RA/Dec with position error
    pos_err = row.get('PosErr')
    ra_triplet = make_triplet(row.get('RAdeg'), pos_err, pos_err)
    dec_triplet = make_triplet(row.get('DEdeg'), pos_err, pos_err)
    
    # Period (orbital period in days)
    period_triplet = make_triplet(row.get('Period'), row.get('Period_err'))
    
    # Eccentricity
    ecc_triplet = make_triplet(row.get('Eccentricity'), row.get('Eccentricity_err'))
    
    # Masses: Mo = companion/accretor (M1), Mx = compact object/past donor (M2)
    m1_triplet = make_triplet(row.get('Mo'), row.get('Mo_err'))
    m2_triplet = make_triplet(row.get('Mx'), row.get('Mx_err'))
    
    # Spectral type and compact object type
    spectype = str(row['Spectype']).strip() if pd.notna(row.get('Spectype')) else None
    compact = str(row['Compact']).strip() if pd.notna(row.get('Compact')) else None
    
    # Map compact object to evolutionary type
    evol_type_2 = map_compact_to_evol_type(compact)
    
    # Override based on compact object mass thresholds
    mx = row.get('Mx')
    if pd.notna(mx):
        mx = float(mx)
        if np.logical_and(mx > 2.5,compact is None):
            # High mass -> likely black hole
            evol_type_2 = "BH"
            compact = "BH?"
        elif mx <= 2.5:
            # Low mass -> likely neutron star
            evol_type_2 = "NS"
            # Add uncertainty to obs_type_2
            if compact is None:
                compact = "NS?"
            elif "?" not in str(compact):
                compact = str(compact) + "?"
    
    # System class: always "high-mass XRB"
    system_class = "high-mass XRB"
    
    # Get Class info for Notes
    class_str = str(row['Class']).strip() if pd.notna(row.get('Class')) else None
    
    # Determine evol_type_1 based on Class
    evol_type_1 = 'MS'  # Default: companion/accretor (high-mass star)
    if class_str and 'WR' in class_str:
        evol_type_1 = 'He-star'  # Wolf-Rayet systems have He-star donors
    
    # Collect all references
    references = collect_references(row)
    
    # Build Notes with Class information if available
    notes = f'High-mass X-ray binary from Fortin et al. 2023 catalog (v2024-08)'
    if class_str:
        notes += f'. Class: {class_str}'
    
    # Build system entry
    system = {
        'System Name': system_name,
        'RA': ra_triplet,
        'Dec': dec_triplet,
        'Period': period_triplet,
        'Eccentricity': ecc_triplet,
        'M1': m1_triplet,
        'M1_sin3i': [None, None, None],
        'M2': m2_triplet,
        'M2_sin3i': [None, None, None],
        'q': [None, None, None],
        'Mass Function': [None, None, None],
        'evol_type_1': evol_type_1,
        'evol_type_2': evol_type_2,
        'obs_type_1': spectype,
        'obs_type_2': compact,
        'system_class': system_class,
        'Detection Method': ['X-ray'],  # All are X-ray binaries
        'Reference': references,
        'Notes': notes,
    }
    
    # Add Simbad link using coordinates
    if ra_triplet[1] is not None and dec_triplet[1] is not None:
        system['Simbad'] = f'https://simbad.cds.unistra.fr/simbad/sim-coo?Coord={ra_triplet[1]}%20{dec_triplet[1]}&Radius=5&Radius.unit=arcsec&output.format=ASCII'
    else:
        system['Simbad'] = None
    
    systems.append(system)

print(f"\nConverted {len(systems)} HMXB systems to JSON schema format")
print(f"Systems with coordinates: {sum(1 for s in systems if s['RA'][1] is not None)}")
print(f"Systems with periods: {sum(1 for s in systems if s['Period'][1] is not None)}")
print(f"Systems with masses: {sum(1 for s in systems if s['M1'][1] is not None or s['M2'][1] is not None)}")
print(f"Systems with spectral types: {sum(1 for s in systems if s['obs_type_1'] is not None)}")
print("\nFirst system example:")
print(json.dumps(systems[0], indent=2))


Converted 164 HMXB systems to JSON schema format
Systems with coordinates: 164
Systems with periods: 115
Systems with masses: 93
Systems with spectral types: 144

First system example:
{
  "System Name": "IGR J00370+6122",
  "RA": [
    2.337165125128296e-09,
    9.290132580203236,
    2.337165125128296e-09
  ],
  "Dec": [
    2.337165125128296e-09,
    61.36013319063004,
    2.337165125128296e-09
  ],
  "Period": [
    0.0014,
    15.6649,
    0.0014
  ],
  "Eccentricity": [
    0.03,
    0.48,
    0.03
  ],
  "M1": [
    null,
    22.0,
    null
  ],
  "M1_sin3i": [
    null,
    null,
    null
  ],
  "M2": [
    null,
    null,
    null
  ],
  "M2_sin3i": [
    null,
    null,
    null
  ],
  "q": [
    null,
    null,
    null
  ],
  "Mass Function": [
    null,
    null,
    null
  ],
  "evol_type_1": "MS",
  "evol_type_2": "NS",
  "obs_type_1": "BN0.7 Ib",
  "obs_type_2": "NS",
  "system_class": "high-mass XRB",
  "Detection Method": [
    "X-ray"
  ],
  "Reference": [
    "2014

In [23]:
# Save to raw JSON file
output_file = Path(RAW_JSON_DIR) / 'Fortin2023_HMXRB.raw.json'
with open(output_file, "w") as f:
    f.write("[\n")
    for i, system in enumerate(systems):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        f.write("  " + line)
        if i < len(systems) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("]\n")

print(f"✓ Saved {len(systems)} HMXB systems to {output_file}")

# Show statistics by system class
from collections import Counter
class_counts = Counter(s['system_class'] for s in systems)
print(f"\nSystem class distribution:")
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls}: {count}")

# Show compact object type distribution
co_counts = Counter(s['evol_type_2'] for s in systems)
print(f"\nCompact object type distribution:")
for co, count in sorted(co_counts.items(), key=lambda x: -x[1]):
    print(f"  {co}: {count}")

✓ Saved 164 HMXB systems to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/Fortin2023_HMXRB.raw.json

System class distribution:
  high-mass XRB: 164

Compact object type distribution:
  NS: 95
  None: 61
  BH: 7
  WD: 1
